# Settings

In [1]:
library(dplyr)
library(gplots)
library(RColorBrewer)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



Attaching package: ‘gplots’


The following object is masked from ‘package:stats’:

    lowess




In [2]:
# Input files
# - rowdata gene table from
rowdata.file <- "../rowdata_human.rds"
# - rowdata column to use for matching
match.col <- "gene_id_base"
# - unformatted expression data 
expr.file <- "../../input_data/GTEx/GTEx_Analysis_v10_RNASeQCv2.4.2_gene_median_tpm.gct.gz"


# Output files
out.files <- list("coldata" = "coldata.rds",
                  "exprmat" = "exprmat.rds",
                  "layout" = "layout.rds")

# Load input data

In [3]:
rowdata <- readRDS(rowdata.file)
nrow(rowdata)
head(rowdata)

[1] 30696

,source,type,gene_id,gene_type,gene_name,level,tag,havana_gene,artif_dupl,gene_id_base,⋯,hom_early_adult,hem_early_adult,sucessful_procedures_13,ortholog,gene_symbol,mean_depmap_gene_effect_score,depmap_essential_05,depmap_essential_1,fusil_all,fusil
,<fct>,<fct>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<int>,<int>,<chr>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>
HGNC:5,HAVANA,gene,ENSG00000121410.12,protein_coding,A1BG,1,overlapping_locus,OTTHUMG00000183507.3,NA,ENSG00000121410,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
HGNC:37133,HAVANA,gene,ENSG00000268895.6,lncRNA,A1BG-AS1,2,overlapping_locus,OTTHUMG00000183508.2,NA,ENSG00000268895,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
HGNC:24086,HAVANA,gene,ENSG00000148584.16,protein_coding,A1CF,2,overlapping_locus,OTTHUMG00000018240.6,NA,ENSG00000148584,⋯,23,0,yes,one2one,A1CF,-0.07773930,n,n,VP,VP
HGNC:7,HAVANA,gene,ENSG00000175899.15,protein_coding,A2M,2,NA,OTTHUMG00000150267.7,NA,ENSG00000175899,⋯,19,0,yes,one2one,A2M,0.02673364,n,n,VP,VP
HGNC:27057,HAVANA,gene,ENSG00000245105.5,lncRNA,A2M-AS1,2,NA,OTTHUMG00000168289.4,NA,ENSG00000245105,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
HGNC:23336,HAVANA,gene,ENSG00000166535.20,protein_coding,A2ML1,2,NA,OTTHUMG00000128499.9,NA,ENSG00000166535,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA


In [4]:
exprdata <- read.delim(expr.file, sep="\t", as.is=T, skip=2)
nrow(exprdata)
head(exprdata)

[1] 59033

,Name,Description,Adipose_Subcutaneous,Adipose_Visceral_Omentum,Adrenal_Gland,Artery_Aorta,Artery_Coronary,Artery_Tibial,Bladder,Brain_Amygdala,⋯,Spleen,Stomach,Stomach_Mixed_Cell,Stomach_Mucosa,Stomach_Muscularis,Testis,Thyroid,Uterus,Vagina,Whole_Blood
,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,ENSG00000223972.5,DDX11L1,0.00000,0.00000,0.000000,0.0000,0.00000,0.00000,0.00000,0.000000,⋯,0.00000,0.000000,0.000000,0.000000,0.0000000,0.167751,0.00000,0.0000,0.00000,0.00000
2,ENSG00000227232.5,WASH7P,3.99789,3.17815,2.673080,4.0708,3.87547,3.62625,5.05094,1.459020,⋯,6.04259,3.041170,3.328970,2.873740,4.0856800,4.542470,6.31791,7.0687,5.74187,2.89503
3,ENSG00000278267.1,MIR6859-1,0.00000,0.00000,0.000000,0.0000,0.00000,0.00000,0.00000,0.000000,⋯,0.00000,0.000000,0.000000,0.000000,0.0000000,0.000000,0.00000,0.0000,0.00000,0.00000
4,ENSG00000243485.5,MIR1302-2HG,0.00000,0.00000,0.000000,0.0000,0.00000,0.00000,0.00000,0.000000,⋯,0.00000,0.000000,0.000000,0.000000,0.0000000,0.054907,0.00000,0.0000,0.00000,0.00000
5,ENSG00000237613.2,FAM138A,0.00000,0.00000,0.000000,0.0000,0.00000,0.00000,0.00000,0.000000,⋯,0.00000,0.000000,0.000000,0.000000,0.0000000,0.000000,0.00000,0.0000,0.00000,0.00000
6,ENSG00000268020.3,OR4G4P,0.00000,0.00000,0.035769,0.0000,0.00000,0.00000,0.00000,0.044141,⋯,0.00000,0.032181,0.038557,0.038184,0.0686565,0.000000,0.00000,0.0000,0.00000,0.00000


In [18]:
# add ensemlb id base (without decimal) for comparison to rowdata
exprdata$Name_base <- exprdata$Name %>%
    lapply(FUN=strsplit, split="\\.") %>%
    unlist() %>%
    matrix(ncol=2, byrow=T)  %>%
    subset.matrix(select = 1) %>%
    as.character()

# remove small number of duplicate ensembl id base vals
temp.dups <- exprdata$Name_base[which(duplicated(exprdata$Name_base))]
exprdata <- exprdata[which(!exprdata$Name_base %in% temp.dups),]
rownames(exprdata) <- exprdata$Name_base
nrow(exprdata)
head(exprdata)

[1] 58943

,Name,Description,Adipose_Subcutaneous,Adipose_Visceral_Omentum,Adrenal_Gland,Artery_Aorta,Artery_Coronary,Artery_Tibial,Bladder,Brain_Amygdala,⋯,Stomach,Stomach_Mixed_Cell,Stomach_Mucosa,Stomach_Muscularis,Testis,Thyroid,Uterus,Vagina,Whole_Blood,Name_base
,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
ENSG00000223972,ENSG00000223972.5,DDX11L1,0.00000,0.00000,0.000000,0.0000,0.00000,0.00000,0.00000,0.000000,⋯,0.000000,0.000000,0.000000,0.0000000,0.167751,0.00000,0.0000,0.00000,0.00000,ENSG00000223972
ENSG00000227232,ENSG00000227232.5,WASH7P,3.99789,3.17815,2.673080,4.0708,3.87547,3.62625,5.05094,1.459020,⋯,3.041170,3.328970,2.873740,4.0856800,4.542470,6.31791,7.0687,5.74187,2.89503,ENSG00000227232
ENSG00000278267,ENSG00000278267.1,MIR6859-1,0.00000,0.00000,0.000000,0.0000,0.00000,0.00000,0.00000,0.000000,⋯,0.000000,0.000000,0.000000,0.0000000,0.000000,0.00000,0.0000,0.00000,0.00000,ENSG00000278267
ENSG00000243485,ENSG00000243485.5,MIR1302-2HG,0.00000,0.00000,0.000000,0.0000,0.00000,0.00000,0.00000,0.000000,⋯,0.000000,0.000000,0.000000,0.0000000,0.054907,0.00000,0.0000,0.00000,0.00000,ENSG00000243485
ENSG00000237613,ENSG00000237613.2,FAM138A,0.00000,0.00000,0.000000,0.0000,0.00000,0.00000,0.00000,0.000000,⋯,0.000000,0.000000,0.000000,0.0000000,0.000000,0.00000,0.0000,0.00000,0.00000,ENSG00000237613
ENSG00000268020,ENSG00000268020.3,OR4G4P,0.00000,0.00000,0.035769,0.0000,0.00000,0.00000,0.00000,0.044141,⋯,0.032181,0.038557,0.038184,0.0686565,0.000000,0.00000,0.0000,0.00000,0.00000,ENSG00000268020


# Make coldata table

In [ ]:
colnames(exprdata)[3:ncol(exprdata)-1]

In [22]:
samples <- colnames(exprdata)[3:(ncol(exprdata)-1)]
coldata <- cbind(samples)
colnames(coldata) <- c("label")
rownames(coldata) <- samples
nrow(coldata)
head(coldata)

[1] 68

,label
Adipose_Subcutaneous,Adipose_Subcutaneous
Adipose_Visceral_Omentum,Adipose_Visceral_Omentum
Adrenal_Gland,Adrenal_Gland
Artery_Aorta,Artery_Aorta
Artery_Coronary,Artery_Coronary
Artery_Tibial,Artery_Tibial


# Make counts table

In [23]:
#match rowdata with exprdata

m <- match(rowdata[,match.col], rownames(exprdata))

# check matches
length(m)
length(which(is.na(m))) # how many NAs?
all(rowdata[which(!is.na(m)),match.col] == rownames(exprdata)[m[which(!is.na(m))]]) # all non-NAs match correctly?

[1] 30696

[1] 994

[1] TRUE

In [24]:
# make exprmat

exprmat <- matrix(NA, nrow=nrow(rowdata), ncol=nrow(coldata))
rownames(exprmat) <- rownames(rowdata)
colnames(exprmat) <- rownames(coldata)
exprmat[,rownames(coldata)]  <- as.matrix(exprdata[m,coldata[rownames(coldata),"label"]])
exprmat <- apply(exprmat, 2, as.numeric)
nrow(exprmat)
head(exprmat)

[1] 30696

Adipose_Subcutaneous,Adipose_Visceral_Omentum,Adrenal_Gland,Artery_Aorta,Artery_Coronary,Artery_Tibial,Bladder,Brain_Amygdala,Brain_Anterior_cingulate_cortex_BA24,Brain_Caudate_basal_ganglia,⋯,Spleen,Stomach,Stomach_Mixed_Cell,Stomach_Mucosa,Stomach_Muscularis,Testis,Thyroid,Uterus,Vagina,Whole_Blood
3.4604200,3.620680,1.503850,3.770370,8.176010,7.127790,1.619620,4.775470,3.993370,4.6752400,⋯,7.062860,1.490460,1.473390,1.130070,2.0906300,1.662100,4.904120,8.341280,5.962760,2.149370
1.4581000,1.753460,0.902609,1.938440,4.515940,3.142660,1.238840,0.922936,1.766090,1.2103800,⋯,5.165660,0.922921,1.026660,0.468293,0.9498620,0.593012,3.436170,5.439510,2.858900,1.237280
0.0074295,0.005686,0.018510,0.016453,0.012284,0.016276,0.010124,0.000000,0.004664,0.0043585,⋯,0.010026,0.245815,0.287449,0.261422,0.0000000,0.047017,0.016468,0.007423,0.010526,0.006940
682.8220000,671.197000,151.002000,1640.330000,1267.960000,1296.910000,789.817000,45.930700,44.279200,62.0376000,⋯,287.727000,174.425000,144.064000,38.140100,230.8470000,61.558800,451.287000,618.916000,253.731000,1.309330
6.8924700,5.314650,5.253860,10.875200,9.820860,9.326200,8.779460,1.453880,1.465070,1.6458000,⋯,7.054070,2.077460,2.578750,0.873060,7.7019400,4.628580,5.060650,14.694800,5.868500,0.225902
0.1148710,0.096511,0.118937,0.173176,0.172619,0.135998,0.303104,1.569770,1.565880,1.8422000,⋯,0.133949,0.125684,0.017108,0.008101,0.0349855,8.607760,0.150964,0.341028,243.086000,0.050016


# Average across replicates

In [ ]:
exprmat.avg <- matrix(NA, nrow=nrow(exprmat), ncol=0)
coldata.avg <- as.data.frame(matrix(NA, nrow=0, ncol=4))
rownames(exprmat.avg) <- rownames(exprmat)
for(t in seq(length(tissues))){
    for(s in seq(length(stages))){
        temp.match <- which(coldata$tissue==tissues[t] & coldata$stage==stages[s])
        if(length(temp.match)==1){
            exprmat.avg <-  cbind(exprmat.avg, exprmat[,temp.match])
            colnames(exprmat.avg)[ncol(exprmat.avg)] <- paste(tissues[t], stages[s], sep="_")}
        if(length(temp.match)>1){
            exprmat.avg <-  cbind(exprmat.avg, rowMeans(exprmat[,temp.match]))
            colnames(exprmat.avg)[ncol(exprmat.avg)] <- paste(tissues[t], stages[s], sep="_")}
        if(length(temp.match)>0){
            coldata.avg <- rbind(coldata.avg, c(paste(tissues[t], stages[s], sep="_"),
                                                tissues[t], stages[s]))}
    }
}
nrow(exprmat.avg)
ncol(exprmat.avg)
head(exprmat.avg)
colnames(coldata.avg) <- c("label", "tissue", "stage")
rownames(coldata.avg) <- coldata.avg$label
nrow(coldata.avg)
head(coldata.avg)

# format layout

In [ ]:
# correct stage labeling to make consitent across tissues
coldata.avg.reformat <- coldata.avg
coldata.avg.reformat$stage[which(coldata.avg.reformat$stage=="Senior")] <- "senior"
coldata.avg.reformat$stage[which(coldata.avg.reformat$stage=="youngTeenager")] <- "school"
coldata.avg.reformat$stage[which(coldata.avg.reformat$stage=="oldTeenager")] <- "teenager"
tissues.reformat <- unique(coldata.avg.reformat$tissue)
stages.reformat <- unique(coldata.avg.reformat$stage)
tissues.reformat
stages.reformat

In [ ]:
# Filter tissues and/or stages in final layout [optional, not done in this case]
# - tissues to keep (all in this case)
tissues.sub <- tissues.reformat
# - stages to keep (all in this case)
stages.sub <- stages.reformat

keep.rows <- c()
for(t in seq(length(tissues.sub))){
    for(s in seq(length(stages.sub))){
        temp.match <- which(coldata.avg.reformat$tissue==tissues.sub[t] & coldata.avg.reformat$stage==stages.sub[s])
        if(length(temp.match)==1){
            keep.rows <- c(keep.rows, temp.match)}
    }
}
coldata.avg.reformat.filt <- coldata.avg.reformat[keep.rows,]
nrow(coldata.avg.reformat.filt)
coldata.avg.reformat.filt

In [ ]:
# order stages for display
length(stages.sub)
stages.sub
stages.sub.ord <- stages.sub[c(1,2,23,3:22)]
length(stages.sub.ord)
stages.sub.ord

In [ ]:
# order tissuesfor display (no change in this case)
length(tissues.sub)
tissues.sub
tissues.sub.ord <- tissues.sub
length(tissues.sub.ord)
tissues.sub.ord

In [ ]:
# plot layout as heatmap
my.cex=1
bias=1
my.pal <- c(brewer.pal(9, "Reds")[c(1,4,8)])
my.cols <- colorRampPalette(my.pal, bias=bias)(100)
layout.heatmap <- matrix(NA, nrow=length(tissues.sub.ord), ncol=length(stages.sub.ord))
for(row in seq(length(tissues.sub.ord))){
    for(col in seq(length(stages.sub.ord))){
        layout.heatmap[row, col] <- length(which(coldata.avg.reformat.filt$tissue==tissues.sub.ord[row] & 
                                                 coldata.avg.reformat.filt$stage==stages.sub.ord[col]))
    }
}
heatmap.2(layout.heatmap, Colv = NA, Rowv = NA, col=my.cols, symbreaks=F,
          tracecol=NA, cexRow=my.cex, cexCol=my.cex, 
          xlab="stages", ylab="tissues", main="How many datasets?",
          labRow=tissues.sub, labCol=stages.sub, key=F, cellnote=round(layout.heatmap, digits=1), notecex=my.cex, notecol="black")

In [ ]:
# create layout matrix by sample name
layout <- matrix(NA, nrow=length(tissues.sub.ord), ncol=length(stages.sub.ord))
rownames(layout) <- tissues.sub.ord
colnames(layout) <- stages.sub.ord
for(row in seq(length(tissues.sub.ord))){
    for(col in seq(length(stages.sub.ord))){
        temp.match <- which(coldata.avg.reformat.filt$tissue==tissues.sub.ord[row] & 
                            coldata.avg.reformat.filt$stage==stages.sub.ord[col])
         if(length(temp.match) == 1){
              layout[row, col] <- coldata.avg.reformat.filt$label[temp.match]}
    }
}
layout

# write output

In [ ]:
saveRDS(coldata.avg, file=out.files[["coldata"]])
saveRDS(exprmat.avg, file=out.files[["exprmat"]])
saveRDS(layout, file=out.files[["layout"]])